# 11 — V4 Unified DEV Combined Feature / Modeling Preregistration Gate

This notebook **does not generate astrology features and does not score any model**.

It verifies that the clean Wave 1 + Wave 2 corpus is frozen, records immutable input hashes, checks chronology/axis composition, freezes the modeling preregistration, and produces the only status that allows a separate feature-generation notebook to be run.


In [1]:

from pathlib import Path
from datetime import datetime
import hashlib, json
import pandas as pd
import numpy as np

NOTEBOOK_VERSION = "SAJU_ML_V4_COMBINED_FEATURE_PREREG_GATE_20260816"

def find_repo_root(start=None):
    p = Path(start or Path.cwd()).resolve()
    for c in [p] + list(p.parents):
        if (c / "saju_engine.py").exists():
            return c
    raise FileNotFoundError("Run from inside the Chartpalja saju repo.")

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024*1024), b""):
            h.update(chunk)
    return h.hexdigest()

def as_bool(s):
    if s.dtype == bool:
        return s
    return s.astype(str).str.lower().map({"true":True,"false":False}).fillna(False)

ROOT = find_repo_root()
W2 = ROOT / "research/ml/artifacts/v4_unified_dev_wave2"
PAIR_PATH = W2 / "V4_UNIFIED_DEV_COMBINED_FROZEN_PAIRS.csv"
AXIS_AUDIT_PATH = W2 / "V4_UNIFIED_DEV_COMBINED_AXIS_AUDIT.csv"
FREEZE_DECISION_PATH = W2 / "V4_UNIFIED_DEV_WAVE2_FREEZE_DECISION.json"

SPEC_PATH = ROOT / "research/ml_corpus/v4_unified_dev_modeling/V4_UNIFIED_DEV_COMBINED_MODELING_PREREGISTRATION.json"
OUT = ROOT / "research/ml/artifacts/v4_unified_dev_modeling_preregistration"
OUT.mkdir(parents=True, exist_ok=True)

for p in [PAIR_PATH, AXIS_AUDIT_PATH, FREEZE_DECISION_PATH, SPEC_PATH]:
    if not p.exists():
        raise FileNotFoundError(p)

pairs = pd.read_csv(PAIR_PATH)
axis_audit = pd.read_csv(AXIS_AUDIT_PATH)
with open(FREEZE_DECISION_PATH, encoding="utf-8") as f:
    freeze = json.load(f)
with open(SPEC_PATH, encoding="utf-8") as f:
    spec = json.load(f)

print("repo:", ROOT)
print("pairs:", len(pairs))
print("spec:", spec["version"])


repo: /Users/sangjinlee/Desktop/projects/saju
pairs: 101
spec: V4_UNIFIED_DEV_COMBINED_MODELING_PREREGISTRATION_V1


## 2. Freeze integrity

In [2]:

EXPECTED_STATUS = "V4_UNIFIED_DEV_WAVE2_FROZEN_COMBINED_TARGETS_MET_READY_FOR_FEATURE_GENERATION_GATE"
assert freeze["status"] == EXPECTED_STATUS
assert freeze["combined"]["all_axis_targets_met"] is True
assert freeze["combined"]["astrology_scored"] is False
assert freeze["wave2"]["astrology_scored"] is False
assert all(v is False for v in freeze["holdout_integrity"].values())

assert len(pairs) == spec["target_corpus"]["required_pairs"] == 101
assert pairs["pair_id"].nunique() == len(pairs)
assert pairs["subject_id"].nunique() == len(pairs), "Preregistered combined corpus expects one pair per subject."
assert (pairs["positive_year"].astype(int) != pairs["negative_year"].astype(int)).all()

if "astrology_scored" in pairs.columns:
    assert as_bool(pairs["astrology_scored"]).sum() == 0

counts = pairs.groupby("axis").size().to_dict()
assert counts == spec["target_corpus"]["required_axis_counts"], (counts, spec["target_corpus"]["required_axis_counts"])

audit_counts = dict(zip(axis_audit["axis"], axis_audit["combined_pairs"].astype(int)))
assert audit_counts == counts
assert axis_audit["target_met"].astype(bool).all()
assert (axis_audit["deficit"].astype(int) == 0).all()

print("Frozen corpus integrity: PASS")
print(counts)


Frozen corpus integrity: PASS
{'COMPETITIVE': 31, 'PROJECT': 25, 'STATUS': 45}


## 3. Pre-model chronology / nuisance diagnostics only

In [3]:

pairs = pairs.copy()
pairs["positive_earlier_calc"] = pairs["positive_year"].astype(int) < pairs["negative_year"].astype(int)
pairs["abs_year_gap_calc"] = (pairs["positive_year"].astype(int) - pairs["negative_year"].astype(int)).abs()

overall = pd.DataFrame([{
    "n_pairs": len(pairs),
    "n_subjects": pairs.subject_id.nunique(),
    "positive_earlier_share": float(pairs.positive_earlier_calc.mean()),
    "same_year_pairs": int((pairs.abs_year_gap_calc == 0).sum()),
    "median_abs_year_gap": float(pairs.abs_year_gap_calc.median()),
    "min_abs_year_gap": int(pairs.abs_year_gap_calc.min()),
    "max_abs_year_gap": int(pairs.abs_year_gap_calc.max()),
}])

axis_diag = (
    pairs.groupby("axis")
    .agg(
        n_pairs=("pair_id","size"),
        positive_earlier_share=("positive_earlier_calc","mean"),
        median_abs_year_gap=("abs_year_gap_calc","median"),
        min_abs_year_gap=("abs_year_gap_calc","min"),
        max_abs_year_gap=("abs_year_gap_calc","max"),
    )
    .reset_index()
)

# Overall chronology is only a nuisance diagnostic. Do not rebalance membership now.
assert 0.40 <= float(overall.loc[0,"positive_earlier_share"]) <= 0.60, (
    "Overall chronology fell outside the preregistration guardrail. "
    "Do NOT directional-backfill; stop and inspect collection process."
)

overall.to_csv(OUT / "V4_COMBINED_PREMODEL_OVERALL_DIAGNOSTIC.csv", index=False)
axis_diag.to_csv(OUT / "V4_COMBINED_PREMODEL_AXIS_DIAGNOSTIC.csv", index=False)

display(overall)
display(axis_diag)


,n_pairs,n_subjects,positive_earlier_share,same_year_pairs,median_abs_year_gap,min_abs_year_gap,max_abs_year_gap
0,101,101,0.564356,0,3.0,1,25


,axis,n_pairs,positive_earlier_share,median_abs_year_gap,min_abs_year_gap,max_abs_year_gap
0,COMPETITIVE,31,0.580645,1.0,1,10
1,PROJECT,25,0.200000,3.0,1,10
2,STATUS,45,0.755556,7.0,1,25


## 4. Freeze model specification and code lineage

In [4]:

engine_path = ROOT / "saju_engine.py"
assert engine_path.exists()

lineage = {
    "created_at": datetime.now().isoformat(timespec="seconds"),
    "notebook_version": NOTEBOOK_VERSION,
    "pair_file": str(PAIR_PATH.relative_to(ROOT)),
    "pair_sha256": sha256_file(PAIR_PATH),
    "axis_audit_sha256": sha256_file(AXIS_AUDIT_PATH),
    "wave2_freeze_decision_sha256": sha256_file(FREEZE_DECISION_PATH),
    "preregistration_spec_sha256": sha256_file(SPEC_PATH),
    "saju_engine_sha256_at_preregistration": sha256_file(engine_path),
    "winner_eligible_architectures": [x["name"] for x in spec["winner_eligible_architectures"]],
    "baselines": [x["name"] for x in spec["baselines"]],
}

with open(OUT / "V4_COMBINED_MODELING_PREREGISTRATION_LINEAGE.json","w",encoding="utf-8") as f:
    json.dump(lineage,f,ensure_ascii=False,indent=2)

print(json.dumps(lineage, ensure_ascii=False, indent=2))


{
  "created_at": "2026-08-16T21:37:20",
  "notebook_version": "SAJU_ML_V4_COMBINED_FEATURE_PREREG_GATE_20260816",
  "pair_file": "research/ml/artifacts/v4_unified_dev_wave2/V4_UNIFIED_DEV_COMBINED_FROZEN_PAIRS.csv",
  "pair_sha256": "5c4cecc76e2059fc6276564b4be13638ffc4b5b11b5773dd5954ed98e7dad31b",
  "axis_audit_sha256": "64d9fa3f7b892ca4efdbd0ad9c20860010efdfbed4e7e336409a3ee70a0792e0",
  "wave2_freeze_decision_sha256": "c13196e525cd1bbfdb4e877eed2a281e632ec417cc355ee42bb8201e8d2a1feb",
  "preregistration_spec_sha256": "de20924dcb3a289d51f999e1518164d0337583d1102e74081ff8a4623d3ce869",
  "saju_engine_sha256_at_preregistration": "d39e0c4d777ae3a19394c9175319a4f8a6a709c2b4d34400965b618ffbfdad1e",
  "winner_eligible_architectures": [
    "TG10_STEM_BRANCH_L2",
    "ALL_L2",
    "ALL_ELASTICNET_AUTO"
  ],
  "baselines": [
    "CHANCE_050",
    "AGE_YOUNGER_FIXED",
    "AGE_LATER_FIXED",
    "NUISANCE_AGE_AXIS",
    "CONTROL_FIXED"
  ]
}


## 5. Gate decision

In [5]:

allowed_arch = [x["name"] for x in spec["winner_eligible_architectures"]]
assert allowed_arch == ["TG10_STEM_BRANCH_L2","ALL_L2","ALL_ELASTICNET_AUTO"]

decision = {
    "version": "V4_UNIFIED_DEV_COMBINED_FEATURE_PREREG_GATE_DECISION_V1",
    "notebook_version": NOTEBOOK_VERSION,
    "created_at": datetime.now().isoformat(timespec="seconds"),
    "status": "V4_COMBINED_FEATURE_PREREGISTERED_READY_FOR_FEATURE_GENERATION",
    "corpus": {
        "n_pairs": int(len(pairs)),
        "n_subjects": int(pairs.subject_id.nunique()),
        "axis_counts": {k:int(v) for k,v in counts.items()},
        "positive_earlier_share": float(pairs.positive_earlier_calc.mean()),
        "astrology_scored": False,
    },
    "winner_eligible_architectures": allowed_arch,
    "production_reference": "CONTROL_FIXED",
    "nuisance_reference": "NUISANCE_AGE_AXIS",
    "promotion_gates": spec["promotion_gates"],
    "holdout_integrity": {
        "NEW_CONFIRM_loaded": False,
        "Validation_B_loaded": False,
        "Public_CHECK_loaded": False,
        "Public_FINAL_loaded": False,
    },
    "forbidden_after_this_gate": [
        "Changing the 101 event pairs or their axis labels",
        "Adding a fourth winner-eligible architecture after seeing results",
        "Hand-tuning feature weights on this corpus",
        "Opening any sealed holdout before a development architecture survives all gates",
    ],
    "next_rule": (
        "Generate the preregistered astrology feature matrix with the recorded engine lineage, "
        "then run a separate nested subject-CV tournament containing only the 3 frozen winner-eligible "
        "architectures plus preregistered baselines. If none survives, do not open a holdout."
    ),
}

with open(OUT / "V4_COMBINED_FEATURE_PREREGISTRATION_DECISION.json","w",encoding="utf-8") as f:
    json.dump(decision,f,ensure_ascii=False,indent=2)

print(json.dumps(decision, ensure_ascii=False, indent=2))


{
  "version": "V4_UNIFIED_DEV_COMBINED_FEATURE_PREREG_GATE_DECISION_V1",
  "notebook_version": "SAJU_ML_V4_COMBINED_FEATURE_PREREG_GATE_20260816",
  "created_at": "2026-08-16T21:37:20",
  "status": "V4_COMBINED_FEATURE_PREREGISTERED_READY_FOR_FEATURE_GENERATION",
  "corpus": {
    "n_pairs": 101,
    "n_subjects": 101,
    "axis_counts": {
      "COMPETITIVE": 31,
      "PROJECT": 25,
      "STATUS": 45
    },
    "positive_earlier_share": 0.5643564356435643,
    "astrology_scored": false
  },
  "winner_eligible_architectures": [
    "TG10_STEM_BRANCH_L2",
    "ALL_L2",
    "ALL_ELASTICNET_AUTO"
  ],
  "production_reference": "CONTROL_FIXED",
  "nuisance_reference": "NUISANCE_AGE_AXIS",
  "promotion_gates": {
    "all_required": true,
    "rules": {
      "overall_subject_macro_pairwise": ">= 0.55",
      "delta_vs_CONTROL_FIXED": ">= 0.01",
      "bootstrap_P_delta_gt_0_vs_CONTROL_FIXED": ">= 0.80",
      "delta_vs_NUISANCE_AGE_AXIS": ">= 0.01",
      "bootstrap_P_delta_gt_0_vs_NUISA

## What to send back

After **Kernel Restart → Run All**, send:

```text
V4_COMBINED_FEATURE_PREREGISTRATION_DECISION.json
V4_COMBINED_MODELING_PREREGISTRATION_LINEAGE.json
V4_COMBINED_PREMODEL_OVERALL_DIAGNOSTIC.csv
V4_COMBINED_PREMODEL_AXIS_DIAGNOSTIC.csv
```

Do not generate astrology features until this gate has passed.
